# Anatomy of a short squeeze: GameStop, 2021

Synthetic reconstruction (seeded). Runs offline.


In [ ]:
import numpy as np, pandas as pd
from scipy.stats import norm
rng = np.random.default_rng(21)


## Synthetic price path with a vertical run-up


In [ ]:
days = pd.bdate_range('2020-12-31', '2021-02-19'); n = len(days)
px = pd.Series(4.0 + rng.normal(0,0.3,n).cumsum()*0, index=days)
px.iloc[:18] = 4 + np.linspace(0, 14, 18)
px.iloc[18:26] = np.linspace(20, 120, 8)
px.iloc[26:] = np.linspace(120, 45, n-26)


## Run-up, realised vol, short P&L


In [ ]:
ret = np.log(px).diff()
run_up = px.max()/px.iloc[0]-1
rvol = ret.rolling(10).std()*np.sqrt(252)
short_pnl = 1 - px/px.iloc[0]
print(f'run-up {run_up:.0%}  worst short {short_pnl.min():.0%}  peak rvol {rvol.max():.0%}')


## Dealer gamma: hedge demand vs price


In [ ]:
def call_delta(S,K,T,r,sigma):
    d1=(np.log(S/K)+(r+0.5*sigma**2)*T)/(sigma*np.sqrt(T)); return norm.cdf(d1)

for S in [40,60,80,100]:
    print(S, round(100*call_delta(S,60,0.05,0,1.2),1), 'shares/contract')
